# 1986 Backpropagation

In 1986, David Rumelhart, Geoffrey Hinton, and Ronald Williams showed how backpropagation could train multilayer neural networks effectively. This made it practical to adjust hidden-layer weights instead of only training a single output perceptron.

## XOR with a Multilayer Perceptron

A single perceptron cannot learn XOR because XOR is not linearly separable. To solve it, we use a small multilayer perceptron with two inputs, one hidden layer, and one output.

The topology below has its own `MultilayerPerceptronNeuron` class. That keeps the single-layer `Perceptron` focused on step activation, while each multilayer neuron owns its sigmoid activation behavior so the trainer can calculate gradients with backpropagation.

Backpropagation starts at the output layer, measures how far the prediction is from the desired output, and sends that error backward through the hidden layer. Each neuron then updates its weights and bias according to how much it contributed to the error.

The trainer code mirrors those steps with separate methods: `train_one_example`, `output_layer_deltas`, `hidden_layer_deltas`, and `update_weights_and_biases`.

For this topology, each neuron uses the sigmoid activation:

$$a = \frac{1}{1 + e^{-z}}$$

The output layer delta is:

$$\delta_{output} = (desired\ output - prediction) \times a(1 - a)$$

Each hidden layer delta uses the deltas from the next layer:

$$\delta_{hidden} = \left(\sum w_{next}\delta_{next}\right) \times a(1 - a)$$

In [3]:
import math
import random


class MultilayerPerceptronNeuron:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

    def weighted_sum(self, inputs):
        return sum(weight * value for weight, value in zip(self.weights, inputs)) + self.bias

    def activate(self, inputs):
        return 1 / (1 + math.exp(-self.weighted_sum(inputs)))

    def activation_derivative(self, activation):
        return activation * (1 - activation)

    def update(self, inputs, delta, learning_rate):
        self.weights = [
            weight + learning_rate * delta * value
            for weight, value in zip(self.weights, inputs)
        ]
        self.bias += learning_rate * delta


class MultilayerPerceptron:
    def __init__(self, layer_sizes, seed=1):
        if len(layer_sizes) < 2:
            raise ValueError("A multilayer perceptron needs at least an input layer and an output layer.")

        random_generator = random.Random(seed)
        self.layers = []

        for input_count, neuron_count in zip(layer_sizes, layer_sizes[1:]):
            layer = [
                MultilayerPerceptronNeuron(
                    weights=[random_generator.uniform(-1, 1) for _ in range(input_count)],
                    bias=random_generator.uniform(-1, 1),
                )
                for _ in range(neuron_count)
            ]
            self.layers.append(layer)

    def forward(self, inputs):
        activations = [list(inputs)]

        for layer in self.layers:
            current_outputs = [neuron.activate(activations[-1]) for neuron in layer]
            activations.append(current_outputs)

        return activations

    def predict_probability(self, inputs):
        return self.forward(inputs)[-1][0]

    def predict(self, inputs):
        return 1 if self.predict_probability(inputs) >= 0.5 else 0


class MultilayerPerceptronTrainer:
    def __init__(self, topology, learning_rate=0.5):
        self.topology = topology
        self.learning_rate = learning_rate

    def train(self, training_data, epochs=10000, target_error=0.01):
        for _ in range(epochs):
            total_error = 0

            for inputs, desired_output in training_data:
                total_error += self.train_one_example(inputs, desired_output)

            if total_error < target_error:
                break

        return self.topology

    def train_one_example(self, inputs, desired_output):
        activations = self.topology.forward(inputs)
        output_errors = self.output_errors(
            desired_outputs=[desired_output],
            predictions=activations[-1],
        )
        deltas = self.backpropagate_deltas(activations, output_errors)
        self.update_weights_and_biases(activations, deltas)

        return sum(error ** 2 for error in output_errors)

    def output_errors(self, desired_outputs, predictions):
        return [
            desired - prediction
            for desired, prediction in zip(desired_outputs, predictions)
        ]

    def backpropagate_deltas(self, activations, output_errors):
        deltas = [None] * len(self.topology.layers)
        deltas[-1] = self.output_layer_deltas(output_errors, activations[-1])

        for layer_index in range(len(self.topology.layers) - 2, -1, -1):
            deltas[layer_index] = self.hidden_layer_deltas(
                layer_index,
                activations,
                deltas[layer_index + 1],
            )

        return deltas

    def output_layer_deltas(self, output_errors, output_activations):
        output_layer = self.topology.layers[-1]

        return [
            error * neuron.activation_derivative(activation)
            for error, activation, neuron in zip(
                output_errors,
                output_activations,
                output_layer,
            )
        ]

    def hidden_layer_deltas(self, layer_index, activations, next_layer_deltas):
        current_layer = self.topology.layers[layer_index]
        next_layer = self.topology.layers[layer_index + 1]
        current_activations = activations[layer_index + 1]

        return [
            self.hidden_neuron_delta(
                neuron_index,
                neuron,
                activation,
                next_layer,
                next_layer_deltas,
            )
            for neuron_index, (neuron, activation) in enumerate(
                zip(current_layer, current_activations)
            )
        ]

    def hidden_neuron_delta(self, neuron_index, neuron, activation, next_layer, next_layer_deltas):
        downstream_error = sum(
            next_neuron.weights[neuron_index] * next_layer_deltas[next_index]
            for next_index, next_neuron in enumerate(next_layer)
        )

        return downstream_error * neuron.activation_derivative(activation)

    def update_weights_and_biases(self, activations, deltas):
        for layer_index, layer in enumerate(self.topology.layers):
            previous_activations = activations[layer_index]

            for neuron, delta in zip(layer, deltas[layer_index]):
                neuron.update(previous_activations, delta, self.learning_rate)


xor_training_data = [
    ((0, 0), 0),
    ((0, 1), 1),
    ((1, 0), 1),
    ((1, 1), 0),
]

xor_gate = MultilayerPerceptron(layer_sizes=[2, 2, 1], seed=1)
MultilayerPerceptronTrainer(xor_gate, learning_rate=0.5).train(xor_training_data)

print("XOR gate")
for values, _ in xor_training_data:
    probability = xor_gate.predict_probability(values)
    print(f"{values} -> {xor_gate.predict(values)} (probability={probability:.3f})")

for inputs, desired_output in xor_training_data:
    assert xor_gate.predict(inputs) == desired_output

print("\nXOR multilayer perceptron checks passed.")

XOR gate
(0, 0) -> 0 (probability=0.050)
(0, 1) -> 1 (probability=0.957)
(1, 0) -> 1 (probability=0.940)
(1, 1) -> 0 (probability=0.044)

XOR multilayer perceptron checks passed.


## Why This Mattered

Backpropagation made it practical to train networks with hidden layers by calculating how each weight contributed to the final error. It turned multilayer neural networks from an interesting idea into a useful learning method, setting up much of the later progress in representation learning and deep learning.